In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import re
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

from bs4 import BeautifulSoup
import re

from spacy.lang.en.stop_words import STOP_WORDS
import en_core_web_sm

In [2]:
data = pd.read_csv('/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_the_north_face/sample-data.csv')

In [3]:
data.head()

,id,description
0,1,Active classic boxers - There's a reason why o...
1,2,Active sport boxer briefs - Skinning up Glory ...
2,3,Active sport briefs - These superbreathable no...
3,4,"Alpine guide pants - Skin in, climb ice, switc..."
4,5,"Alpine wind jkt - On high ridges, steep ice an..."


In [4]:
# sample = data.sample(n=5, random_state=42)
sample = data.copy()

In [5]:
pd.set_option('display.max_colwidth', None)
display(sample)

,id,description
0,1,"Active classic boxers - There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Silky Capilene 1 fabric is ultralight, breathable and quick-to-dry""</li> <li>""Exposed, brushed elastic waistband for comfort""</li> <li>5-panel construction with traditional boxer back</li> <li>""Inseam (size M) is 4 1/2""""""</li></ul><br><br><b>Fabric: </b>3.7-oz 100% all-recycled polyester with Gladiodor natural odor control for the garment. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>99 g (3.5 oz)<br><br>Made in Mexico."
1,2,"Active sport boxer briefs - Skinning up Glory requires enough movement without your boxers deciding to poach their own route. The form-fitting Active Sport Boxer Briefs are made from breathable 93% polyester (71% recycled) fabric that's fast-wicking, dries quickly and has 7% spandex for stretch; the seamless waistband and soft leg edges won't roll or bind. The gusseted, flat-sewn 6"" inseam (size M) is offset to prevent inner-thigh chafe. Fly-free with a smooth front panel. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast""</li> <li>Seamless construction</li> <li>""Flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe""</li> <li>Fly-free support</li> <li>""Inseam (size M) is 6""""""</li></ul><br><br><b>Fabric: </b>""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""<br><br><b>Weight: </b>(60 g 2.1 oz)<br><br>Made in Israel."
2,3,"Active sport briefs - These superbreathable no-fly briefs are the minimalist's choice for high-octane endeavors. Made from a blend of fast-wicking, quick-drying 93% polyester (71% recycled) and 7% spandex that has both stretch-mesh (for support) and open mesh (for cooling airflow). Soft edging at the leg openings and a seamless waist won't roll or create friction against layers. With a smooth front panel for opacity. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently and dries fast""</li> <li>Seamless construction</li> <li>Leg openings have half-inch self-binding for comfort</li> <li>Fly-free support</li></ul><br><br><b>Fabric: </b>""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""<br><br><b>Weight: </b>(49 g 1.7 oz)<br><br>Made in Israel."
3,4,"Alpine guide pants - Skin in, climb ice, switch to rock, traverse a knife-edge ridge and boogie back down - these durable, weather-resistant and breathable soft-shell pants keep stride on every mountain endeavor. The midweight stretch-woven polyester won't restrict your moves, and the brushed interior maintains next-to-skin comfort. A 2-way zippered fly keeps things easy with a harness on, and the gusseted, zippered cuffs have 2-position snaps and tie-down loops. With water-resistant zippered pockets: two front, two thigh, one back hip. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>Durable stretch-woven polyester (47% recycled) with DWR (durable water repellent) finish is water- and wind-resistant and highly breathable</li> <li>Brushed interior for next-to-skin comfort</li> <li>Waistband has belt loops and elastic on back</li> <li>2-way zippered fly

In [6]:
sample_clean = sample.copy()

# Créer un objet BeautifulSoup pour chaque description
html = sample_clean["description"].fillna("").apply(
    lambda texte: BeautifulSoup(texte, "html.parser")
)

In [7]:
def extraire_description(soup):
    """Récupérer le texte avant la première balise <br> ou <b>."""
    morceaux = []

    for element in soup.contents:
        if element.name in ["br", "b"]:
            break

        morceaux.append(str(element))

    texte_html = " ".join(morceaux)
    return BeautifulSoup(texte_html, "html.parser").get_text(" ", strip=True)


sample_clean["description_texte"] = html.apply(extraire_description)

In [8]:
def extraire_details(soup):
    """Récupérer les éléments de la liste Details."""
    titre = soup.find("b", string=re.compile(r"^\s*Details\s*:\s*$", re.I))

    if titre is None:
        return []

    liste = titre.find_next_sibling()

    if liste is None or liste.name != "ul":
        return []

    details = []

    for element in liste.find_all("li"):
        details.append(element.get_text(" ", strip=True))

    return details


sample_clean["details"] = html.apply(extraire_details)

In [9]:
def extraire_valeur(soup, nom):
    """Récupérer le texte qui suit un titre en gras."""
    motif = rf"^\s*{re.escape(nom)}\s*:\s*$"
    titre = soup.find("b", string=re.compile(motif, re.I))

    if titre is None:
        return None

    morceaux = []

    for element in titre.next_siblings:
        if element.name in ["br", "b"]:
            break

        morceaux.append(str(element))

    texte_html = " ".join(morceaux)
    return BeautifulSoup(texte_html, "html.parser").get_text(" ", strip=True)


sample_clean["fabric"] = html.apply(lambda soup: extraire_valeur(soup, "Fabric"))
sample_clean["weight"] = html.apply(lambda soup: extraire_valeur(soup, "Weight"))

In [10]:
def extraire_pays(soup):
    """Récupérer le pays après « Made in »."""
    for texte in soup.stripped_strings:
        if texte.lower().startswith("made in "):
            return texte[8:].strip().rstrip(".")

    return None


sample_clean["pays"] = html.apply(extraire_pays)

In [11]:
display(sample_clean[[
    "description_texte",
    "details",
    "fabric",
    "weight",
    "pays"
]])

,description_texte,details,fabric,weight,pays
0,"Active classic boxers - There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.","[""Silky Capilene 1 fabric is ultralight, breathable and quick-to-dry"", ""Exposed, brushed elastic waistband for comfort"", 5-panel construction with traditional boxer back, ""Inseam (size M) is 4 1/2""""""]",3.7-oz 100% all-recycled polyester with Gladiodor natural odor control for the garment. Recyclable through the Common Threads Recycling Program,99 g (3.5 oz),Mexico
1,"Active sport boxer briefs - Skinning up Glory requires enough movement without your boxers deciding to poach their own route. The form-fitting Active Sport Boxer Briefs are made from breathable 93% polyester (71% recycled) fabric that's fast-wicking, dries quickly and has 7% spandex for stretch; the seamless waistband and soft leg edges won't roll or bind. The gusseted, flat-sewn 6"" inseam (size M) is offset to prevent inner-thigh chafe. Fly-free with a smooth front panel. Recyclable through the Common Threads Recycling Program.","[""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast"", Seamless construction, ""Flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe"", Fly-free support, ""Inseam (size M) is 6""""""]","""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""",(60 g 2.1 oz),Israel
2,"Active sport briefs - These superbreathable no-fly briefs are the minimalist's choice for high-octane endeavors. Made from a blend of fast-wicking, quick-drying 93% polyester (71% recycled) and 7% spandex that has both stretch-mesh (for support) and open mesh (for cooling airflow). Soft edging at the leg openings and a seamless waist won't roll or create friction against layers. With a smooth front panel for opacity. Recyclable through the Common Threads Recycling Program.","[""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently and dries fast"", Seamless construction, Leg openings have half-inch self-binding for comfort, Fly-free support]","""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""",(49 g 1.7 oz),Israel
3,"Alpine guide pants - Skin in, climb ice, switch to rock, traverse a knife-edge ridge and boogie back down - these durable, weather-resistant and breathable soft-shell pants keep stride on every mountain endeavor. The midweight stretch-woven polyester won't restrict your moves, and the brushed interior maintains next-to-skin comfort. A 2-way zippered fly keeps things easy with a harness on, and the gusseted, zippered cuffs have 2-position snaps and tie-down loops. With water-resistant zippered pockets: two front, two thigh, one back hip. Recyclable through the Common Threads Recycling Program.","[Durable stretch-woven polyester (47% recycled) with DWR (durable water repellent) finish is water- and wind-resistant and highly breathable, Brushed interior for next-to-skin comfort, Waistband has belt loops and elastic on back, 2-way zippered fly, ""External pockets: two front slash, two glued-on thigh, one back hip - all with highly water-resistant, DWR-finished zippers"", ""Gusseted, zippered cuff with 2-position adjustable settings and tie-down loops""]","""7.6-oz 90-denier 92% polyester (47% recycled)/8% spandex, with Deluge DWR finish. Recyclable through the Common Threads Recycling Program""",(597 g 20.7 oz),Vietnam
4,"Alpine win

In [12]:
sample_clean["details"] = sample_clean["details"].apply(
    lambda liste: " ".join(liste)
)

In [13]:
def nettoyer_texte(texte):
    texte = texte.lower()
    texte = re.sub(r"[^\w\s%/.,-]", " ", texte)
    return re.sub(r"\s+", " ", texte).strip()

In [14]:
for col in ["description_texte", "details", "fabric", "weight"]:
    sample_clean[col] = sample_clean[col].apply(
        lambda texte: nettoyer_texte(texte) if pd.notna(texte) else texte
    )

In [15]:
sample_clean.shape

(500, 7)

In [16]:
display(sample_clean)

,id,description,description_texte,details,fabric,weight,pays
0,1,"Active classic boxers - There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Silky Capilene 1 fabric is ultralight, breathable and quick-to-dry""</li> <li>""Exposed, brushed elastic waistband for comfort""</li> <li>5-panel construction with traditional boxer back</li> <li>""Inseam (size M) is 4 1/2""""""</li></ul><br><br><b>Fabric: </b>3.7-oz 100% all-recycled polyester with Gladiodor natural odor control for the garment. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>99 g (3.5 oz)<br><br>Made in Mexico.","active classic boxers - there s a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. the quick-drying, lightweight underwear takes up minimal space in a travel pack. an exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. made of 3.7-oz 100% recycled polyester with moisture-wicking performance. inseam size m is 4 1/2 . recyclable through the common threads recycling program.","silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2",3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program,99 g 3.5 oz,Mexico
1,2,"Active sport boxer briefs - Skinning up Glory requires enough movement without your boxers deciding to poach their own route. The form-fitting Active Sport Boxer Briefs are made from breathable 93% polyester (71% recycled) fabric that's fast-wicking, dries quickly and has 7% spandex for stretch; the seamless waistband and soft leg edges won't roll or bind. The gusseted, flat-sewn 6"" inseam (size M) is offset to prevent inner-thigh chafe. Fly-free with a smooth front panel. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast""</li> <li>Seamless construction</li> <li>""Flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe""</li> <li>Fly-free support</li> <li>""Inseam (size M) is 6""""""</li></ul><br><br><b>Fabric: </b>""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""<br><br><b>Weight: </b>(60 g 2.1 oz)<br><br>Made in Israel.","active sport boxer briefs - skinning up glory requires enough movement without your boxers deciding to poach their own route. the form-fitting active sport boxer briefs are made from breathable 93% polyester 71% recycled fabric that s fast-wicking, dries quickly and has 7% spandex for stretch the seamless waistband and soft leg edges won t roll or bind. the gusseted, flat-sewn 6 inseam size m is offset to prevent inner-thigh chafe. fly-free with a smooth front panel. recyclable through the common threads recycling program.","stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast seamless construction flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe fly-free support inseam size m is 6","4.6-oz 93% polyester 71% recycled /7% spandex, with moisture-wicking performance. recyclable through the common threads recycling program",60 g 2.1 oz,Israel
2,3,"Active sport briefs - These superb

In [17]:
sample_clean.drop(columns=["description"], inplace=True)

In [18]:
sample_clean

,id,description_texte,details,fabric,weight,pays
0,1,"active classic boxers - there s a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. the quick-drying, lightweight underwear takes up minimal space in a travel pack. an exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. made of 3.7-oz 100% recycled polyester with moisture-wicking performance. inseam size m is 4 1/2 . recyclable through the common threads recycling program.","silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2",3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program,99 g 3.5 oz,Mexico
1,2,"active sport boxer briefs - skinning up glory requires enough movement without your boxers deciding to poach their own route. the form-fitting active sport boxer briefs are made from breathable 93% polyester 71% recycled fabric that s fast-wicking, dries quickly and has 7% spandex for stretch the seamless waistband and soft leg edges won t roll or bind. the gusseted, flat-sewn 6 inseam size m is offset to prevent inner-thigh chafe. fly-free with a smooth front panel. recyclable through the common threads recycling program.","stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast seamless construction flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe fly-free support inseam size m is 6","4.6-oz 93% polyester 71% recycled /7% spandex, with moisture-wicking performance. recyclable through the common threads recycling program",60 g 2.1 oz,Israel
2,3,"active sport briefs - these superbreathable no-fly briefs are the minimalist s choice for high-octane endeavors. made from a blend of fast-wicking, quick-drying 93% polyester 71% recycled and 7% spandex that has both stretch-mesh for support and open mesh for cooling airflow . soft edging at the leg openings and a seamless waist won t roll or create friction against layers. with a smooth front panel for opacity. recyclable through the common threads recycling program.","stretch mesh provides support, open-weave mesh for airflow, wicks efficiently and dries fast seamless construction leg openings have half-inch self-binding for comfort fly-free support","4.6-oz 93% polyester 71% recycled /7% spandex, with moisture-wicking performance. recyclable through the common threads recycling program",49 g 1.7 oz,Israel
3,4,"alpine guide pants - skin in, climb ice, switch to rock, traverse a knife-edge ridge and boogie back down - these durable, weather-resistant and breathable soft-shell pants keep stride on every mountain endeavor. the midweight stretch-woven polyester won t restrict your moves, and the brushed interior maintains next-to-skin comfort. a 2-way zippered fly keeps things easy with a harness on, and the gusseted, zippered cuffs have 2-position snaps and tie-down loops. with water-resistant zippered pockets two front, two thigh, one back hip. recyclable through the common threads recycling program.","durable stretch-woven polyester 47% recycled with dwr durable water repellent finish is water- and wind-resistant and highly breathable brushed interior for next-to-skin comfort waistband has belt loops and elastic on back 2-way zippered fly external pockets two front slash, two glued-on thigh, one back hip - all with highly water-resistant, dwr-finished zippers gusseted, zippered cuff with 2-position adjustable settings and tie-down loops","7.6-oz 90-denier 92% polyester 47% recycled /8% spandex, with deluge dwr finish. recyclable through the common threads recycling program",597 g 20.7 oz,Vietnam
4,5,"alpine wind jkt - on high ridges, steep ice and anything alpine, this jacket serves as a true best of all worlds

In [19]:
nlp = en_core_web_sm.load()

for col in ["description_texte", "details", "fabric", "weight"]:
    sample_clean[col] = sample_clean[col].apply(
        lambda texte: " ".join(
            token.text for token in nlp(texte) if not token.is_stop
        ) if pd.notna(texte) else texte
    )

In [20]:
for col in ["description_texte", "details", "fabric", "weight"]:
    print(f"\n--- {col} ---")
    for texte in sample_clean[col].dropna():
        doc = nlp(texte)
        print("Tokens:", [token.text for token in doc])
        print("Lemma:", [token.lemma_ for token in doc])
        print("Indices:", [token.i for token in doc])
        print("Alpha Numeric:", [token.is_alpha for token in doc])
        print("Numbers:", [token.like_num for token in doc])
        print("Stopwords restants:", [token.text for token in doc if token.is_stop])
        print("------------")


--- description_texte ---
Tokens: ['active', 'classic', 'boxers', '-', 's', 'reason', 'boxers', 'cult', 'favorite', '-', 'cool', ',', 'especially', 'sticky', 'situations', '.', 'quick', '-', 'drying', ',', 'lightweight', 'underwear', 'takes', 'minimal', 'space', 'travel', 'pack', '.', 'exposed', ',', 'brushed', 'waistband', 'offers', '-', '-', 'skin', 'softness', ',', '-', 'panel', 'construction', 'traditional', 'boxer', 'classic', 'fit', ',', 'functional', 'fly', '.', '3.7', '-', 'oz', '100', '%', 'recycled', 'polyester', 'moisture', '-', 'wicking', 'performance', '.', 'inseam', 'size', 'm', '4', '1/2', '.', 'recyclable', 'common', 'threads', 'recycling', 'program', '.']
Lemma: ['active', 'classic', 'boxer', '-', 's', 'reason', 'boxer', 'cult', 'favorite', '-', 'cool', ',', 'especially', 'sticky', 'situation', '.', 'quick', '-', 'dry', ',', 'lightweight', 'underwear', 'take', 'minimal', 'space', 'travel', 'pack', '.', 'expose', ',', 'brush', 'waistband', 'offer', '-', '-', 'skin', 's

In [21]:
print(STOP_WORDS)

{'five', 'become', '‘d', 'per', 'keep', 'every', 'done', 'against', 'hereupon', 'seeming', 'hereafter', 'top', 'how', 'may', 'none', 'had', 'have', 'for', 'sometimes', 'ever', 'latter', 'four', 'moreover', '‘re', 'only', 'full', 'towards', 'via', 'yourself', 'whereafter', 'through', 'seem', 'themselves', 'becomes', 'on', 'again', 'well', 'indeed', 'n‘t', 'several', 'therefore', 'us', 'all', 'both', 'further', 'i', '’m', 'say', 'whom', 'before', 'which', 'who', 'around', 'alone', 'quite', 'and', 'serious', '‘ve', 'in', 'anyhow', 'cannot', 'thereby', 'could', 'once', 'hundred', 'same', 'whenever', 'beforehand', 'bottom', 'back', 'an', '‘s', 'above', 'see', 'been', '’s', 'between', 'no', 'nevertheless', 'nor', 'anyway', 'somewhere', 'very', 'one', 'own', 'across', 'here', 'noone', 'our', 'she', 'because', 'everything', 'third', 'else', 'where', 'made', 'yours', 'nothing', 'the', 'beyond', 'behind', 'were', 'neither', 'they', 'something', 'twenty', 'wherein', 'call', 're', 'put', 'unless',

In [22]:
display(sample_clean)

,id,description_texte,details,fabric,weight,pays
0,1,"active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program .","silky capilene 1 fabric ultralight , breathable quick - - dry exposed , brushed elastic waistband comfort 5 - panel construction traditional boxer inseam size m 4 1/2",3.7 - oz 100 % - recycled polyester gladiodor natural odor control garment . recyclable common threads recycling program,99 g 3.5 oz,Mexico
1,2,"active sport boxer briefs - skinning glory requires movement boxers deciding poach route . form - fitting active sport boxer briefs breathable 93 % polyester 71 % recycled fabric s fast - wicking , dries quickly 7 % spandex stretch seamless waistband soft leg edges won t roll bind . gusseted , flat - sewn 6 inseam size m offset prevent inner - thigh chafe . fly - free smooth panel . recyclable common threads recycling program .","stretch mesh provides support , open - weave mesh airflow , wicks efficiently , dries fast seamless construction flat - sewn , gusseted inseam set forward prevent inner - thigh chafe fly - free support inseam size m 6","4.6 - oz 93 % polyester 71 % recycled /7 % spandex , moisture - wicking performance . recyclable common threads recycling program",60 g 2.1 oz,Israel
2,3,"active sport briefs - superbreathable - fly briefs minimalist s choice high - octane endeavors . blend fast - wicking , quick - drying 93 % polyester 71 % recycled 7 % spandex stretch - mesh support open mesh cooling airflow . soft edging leg openings seamless waist won t roll create friction layers . smooth panel opacity . recyclable common threads recycling program .","stretch mesh provides support , open - weave mesh airflow , wicks efficiently dries fast seamless construction leg openings half - inch self - binding comfort fly - free support","4.6 - oz 93 % polyester 71 % recycled /7 % spandex , moisture - wicking performance . recyclable common threads recycling program",49 g 1.7 oz,Israel
3,4,"alpine guide pants - skin , climb ice , switch rock , traverse knife - edge ridge boogie - durable , weather - resistant breathable soft - shell pants stride mountain endeavor . midweight stretch - woven polyester won t restrict moves , brushed interior maintains - - skin comfort . 2 - way zippered fly keeps things easy harness , gusseted , zippered cuffs 2 - position snaps tie - loops . water - resistant zippered pockets , thigh , hip . recyclable common threads recycling program .","durable stretch - woven polyester 47 % recycled dwr durable water repellent finish water- wind - resistant highly breathable brushed interior - - skin comfort waistband belt loops elastic 2 - way zippered fly external pockets slash , glued - thigh , hip - highly water - resistant , dwr - finished zippers gusseted , zippered cuff 2 - position adjustable settings tie - loops","7.6 - oz 90 - denier 92 % polyester 47 % recycled /8 % spandex , deluge dwr finish . recyclable common threads recycling program",597 g 20.7 oz,Vietnam
4,5,"alpine wind jkt - high ridges , steep ice alpine , jacket serves true best worlds staple . excels stand - shell blustery rock climbs , cool - weather trail runs high - output ski tours . , conditions ice alpine climbing , functions lightly insulated windshirt approach , frictionless midlayer s time bundle tie - . polyester ripstop shell deluge dwr durable water repellent treatment sheds snow blocks wind , smooth , lightly brushed hanging mesh liner wicks moisture , dries fast , doesn t bind baselayers . superlight stretch - woven underarm panels enhance breathability allow unimpaired arm motion , hand pockets close zippers . drawcord hem , elastic cuffs

In [24]:
sample_clean.shape

(500, 6)

In [23]:
sample_clean.to_csv('/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_the_north_face/Nico/sample_clean.csv', index=False)